# CIFAR Random Test Infusion - Results Analysis Notebook

This notebook analyzes results from the random test infusion experiment.
It can be run while the experiment is running to monitor progress.

In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from glob import glob
from collections import defaultdict
import pandas as pd
import seaborn as sns

## Configuration

In [2]:
RESULTS_DIR = './results/random_test_infusion/'
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Analyzing results from: {RESULTS_DIR}")

Analyzing results from: ./results/random_test_infusion/


## Visualizations

In [3]:
def load_experiment_result(exp_dir):
    """Load a single experiment result"""
    result_path = os.path.join(exp_dir, 'results.npz')
    if os.path.exists(result_path):
        return np.load(result_path, allow_pickle=True)
    return None


## Load Individual Results

In [4]:
# glob is a standard module for file pattern matching
from glob import glob
import os
# Find all experiment directories
exp_dirs = sorted(glob(os.path.join(RESULTS_DIR, 'sample_*')))
print(f"\n{len(exp_dirs)} experiment directories found")

if len(exp_dirs) > 0:
    # Load a sample result to demonstrate structure
    print("\nLoading sample result...")
    sample_result = load_experiment_result(exp_dirs[0])

    if sample_result is not None:
        print("\nAvailable keys in result:")
        for key in sample_result.keys():
            value = sample_result[key]
            if isinstance(value, np.ndarray):
                print(f"  {key}: shape = {value.shape}, dtype = {value.dtype}")
            else:
                print(f"  {key}: {type(value)}")


685 experiment directories found

Loading sample result...

Available keys in result:
  sample_idx: shape = (), dtype = int64
  test_image_idx: shape = (), dtype = int64
  true_label: shape = (), dtype = int64
  target_class: shape = (), dtype = int64
  probe_image: shape = (3, 32, 32), dtype = float32
  influence_scores: shape = (45000,), dtype = float32
  top_k_indices: shape = (100,), dtype = int64
  selected_train_indices: shape = (100,), dtype = int64
  original_train_images: shape = (100, 3, 32, 32), dtype = float32
  original_train_labels: shape = (100,), dtype = int64
  perturbed_train_images: shape = (100, 3, 32, 32), dtype = float32
  perturbation_norms: shape = (100,), dtype = float32
  logits_epoch9: shape = (1, 10), dtype = float32
  logits_epoch10: shape = (1, 10), dtype = float32
  logits_infused: shape = (1, 10), dtype = float32
  logits_orig_on_selected: shape = (100, 10), dtype = float32
  logits_orig_on_perturbed: shape = (100, 10), dtype = float32
  logits_infused_

# Entropy and Targeted Change Analysis

This section analyzes whether infusion creates **targeted** changes to the target class probability, or whether it simply increases model entropy (uncertainty). We'll examine:

1. **Entropy changes**: Does entropy increase or decrease after infusion?
2. **Target class specificity**: Does the target class increase MORE than other classes?
3. **Probability mass flow**: Where does the probability come from when target increases?
4. **Control comparisons**: Do successful infusions behave differently from unsuccessful ones?

In [5]:
# Load all experiment results and compute entropy for each
import scipy.stats

def compute_entropy(probs):
    """Compute Shannon entropy from probability distribution"""
    # Filter out zero probabilities to avoid log(0)
    probs_nonzero = probs[probs > 0]
    return -np.sum(probs_nonzero * np.log(probs_nonzero))

if len(exp_dirs) > 0:
    print("Loading results and computing entropy...")
    
    entropy_data = []
    
    for exp_dir in exp_dirs:
        result = load_experiment_result(exp_dir)
        if result is None:
            continue
            
        # Get logits
        logits_orig = result['logits_epoch10'][0]
        logits_inf = result['logits_infused'][0]
        
        # Convert to probabilities
        probs_orig = np.exp(logits_orig - np.max(logits_orig))
        probs_orig /= probs_orig.sum()
        
        probs_inf = np.exp(logits_inf - np.max(logits_inf))
        probs_inf /= probs_inf.sum()
        
        # Compute entropies
        entropy_orig = compute_entropy(probs_orig)
        entropy_inf = compute_entropy(probs_inf)
        delta_entropy = entropy_inf - entropy_orig
        
        # Get class probabilities
        true_label = int(result['true_label'])
        target_class = int(result['target_class'])
        
        prob_target_orig = probs_orig[target_class]
        prob_target_inf = probs_inf[target_class]
        delta_p_target = prob_target_inf - prob_target_orig
        
        prob_true_orig = probs_orig[true_label]
        prob_true_inf = probs_inf[true_label]
        delta_p_true = prob_true_inf - prob_true_orig
        
        # Store all probability changes
        delta_probs_all = probs_inf - probs_orig
        
        entropy_data.append({
            'sample_idx': int(result['sample_idx']),
            'test_image_idx': int(result['test_image_idx']),
            'true_label': true_label,
            'target_class': target_class,
            'entropy_orig': entropy_orig,
            'entropy_inf': entropy_inf,
            'delta_entropy': delta_entropy,
            'prob_target_orig': prob_target_orig,
            'prob_target_inf': prob_target_inf,
            'delta_p_target': delta_p_target,
            'prob_true_orig': prob_true_orig,
            'prob_true_inf': prob_true_inf,
            'delta_p_true': delta_p_true,
            'probs_orig': probs_orig,
            'probs_inf': probs_inf,
            'delta_probs_all': delta_probs_all,
            'predicted_orig': np.argmax(probs_orig),
            'predicted_inf': np.argmax(probs_inf),
        })
    
    df_entropy = pd.DataFrame(entropy_data)
    df_entropy['success'] = df_entropy['predicted_inf'] == df_entropy['target_class']
    
    print(f"Loaded {len(df_entropy)} experiments with entropy data")
    print(f"\nEntropy statistics:")
    print(f"  Original model entropy: {df_entropy['entropy_orig'].mean():.4f} ± {df_entropy['entropy_orig'].std():.4f}")
    print(f"  Infused model entropy:  {df_entropy['entropy_inf'].mean():.4f} ± {df_entropy['entropy_inf'].std():.4f}")
    print(f"  Δ entropy (mean):       {df_entropy['delta_entropy'].mean():+.4f} ± {df_entropy['delta_entropy'].std():.4f}")
    print(f"  Δ entropy (median):     {df_entropy['delta_entropy'].median():+.4f}")
    
    # Analyze by direction
    n_increased = (df_entropy['delta_entropy'] > 0).sum()
    n_decreased = (df_entropy['delta_entropy'] < 0).sum()
    n_unchanged = (df_entropy['delta_entropy'] == 0).sum()
    
    print(f"\nEntropy change direction:")
    print(f"  Increased: {n_increased} ({n_increased/len(df_entropy)*100:.1f}%)")
    print(f"  Decreased: {n_decreased} ({n_decreased/len(df_entropy)*100:.1f}%)")
    print(f"  Unchanged: {n_unchanged} ({n_unchanged/len(df_entropy)*100:.1f}%)")
else:
    df_entropy = pd.DataFrame()
    print("No experiment directories found")

Loading results and computing entropy...


Loaded 684 experiments with entropy data

Entropy statistics:
  Original model entropy: 0.5095 ± 0.4440
  Infused model entropy:  0.8778 ± 0.5033
  Δ entropy (mean):       +0.3684 ± 0.5439
  Δ entropy (median):     +0.4134

Entropy change direction:
  Increased: 538 (78.7%)
  Decreased: 146 (21.3%)
  Unchanged: 0 (0.0%)


In [6]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.linear_model import LinearRegression

# Center entropy
delta_entropy_centered = df_entropy['delta_entropy'] - df_entropy['delta_entropy'].mean()
y_target = df_entropy['delta_p_target']

# Fit regression
model = LinearRegression()
model.fit(delta_entropy_centered.values.reshape(-1,1), y_target)
pred_line = model.predict(delta_entropy_centered.values.reshape(-1,1))

# Prepare sorted values for plotting fitted line
delta_entropy_sorted = np.sort(delta_entropy_centered.values)
fitted_sorted = model.predict(delta_entropy_sorted.reshape(-1, 1))

# Create combined figure using FigureWidget for interactivity
fig = go.FigureWidget(make_subplots(rows=1, cols=2, 
                    column_widths=[0.55, 0.45],
                    horizontal_spacing=0.12,
                    subplot_titles=("Δp(Target) vs ΔEntropy", "Probability Distribution")))

# --- LEFT PANEL: Regression scatter + line ---
fig.add_trace(
    go.Scatter(
        x=delta_entropy_centered,
        y=y_target,
        mode='markers',
        name='Data points',
        marker=dict(size=6, color='steelblue', opacity=0.7),
        customdata=np.arange(len(df_entropy)),  # keep index to identify clicked point
        hovertemplate="Index %{customdata}<br>ΔEntropy=%{x:.3f}<br>Δp(Target)=%{y:.3f}<extra></extra>"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=delta_entropy_sorted,
        y=fitted_sorted,
        mode='lines',
        name='Fitted line',
        line=dict(color='red')
    ),
    row=1, col=1
)

fig.update_xaxes(title_text="ΔEntropy (centered)", row=1, col=1)
fig.update_yaxes(title_text="Δp(Target)", row=1, col=1)

# --- RIGHT PANEL: bar chart (starts with first example) ---
CLASS_NAMES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Initialize with first example
first_row = df_entropy.iloc[0]
probs_orig_init = first_row['probs_orig']
probs_inf_init = first_row['probs_inf']

fig.add_trace(go.Bar(x=CLASS_NAMES, y=probs_orig_init, name='Original (epoch 10)', 
                     marker_color='steelblue'), row=1, col=2)
fig.add_trace(go.Bar(x=CLASS_NAMES, y=probs_inf_init, name='Infused', 
                     marker_color='coral'), row=1, col=2)

# Add markers for true label and target class
true_label_init = first_row['true_label']
target_class_init = first_row['target_class']
max_prob_init = max(probs_inf_init.max(), probs_orig_init.max())

fig.add_trace(go.Scatter(x=[CLASS_NAMES[true_label_init]], y=[max_prob_init * 1.1], 
                         mode='markers+text', marker=dict(color='green', size=12, symbol='triangle-down'),
                         text=['True'], textposition='top center', name='True label',
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=[CLASS_NAMES[target_class_init]], y=[max_prob_init * 1.1], 
                         mode='markers+text', marker=dict(color='red', size=12, symbol='triangle-down'),
                         text=['Target'], textposition='top center', name='Target class',
                         showlegend=False), row=1, col=2)

fig.update_yaxes(title_text="Probability", row=1, col=2)
fig.update_layout(barmode='group', height=500, width=1100, showlegend=True, 
                  margin=dict(r=50))

# --- Define callback to update bar chart on click ---
def update_bar_chart(trace, points, selector):
    if not points.point_inds:
        return
    
    idx = points.point_inds[0]
    row = df_entropy.iloc[idx]
    
    # Get probability distributions
    probs_orig = row['probs_orig']
    probs_inf = row['probs_inf']
    true_label = row['true_label']
    target_class = row['target_class']
    
    # Update bar chart data (traces 2 and 3 are the bars)
    with fig.batch_update():
        fig.data[2].y = probs_orig
        fig.data[3].y = probs_inf
        
        # Update markers (traces 4 and 5)
        max_prob = max(probs_inf.max(), probs_orig.max())
        fig.data[4].x = [CLASS_NAMES[true_label]]
        fig.data[4].y = [max_prob * 1.1]
        fig.data[5].x = [CLASS_NAMES[target_class]]
        fig.data[5].y = [max_prob * 1.1]
        
        # Update y-axis range to accommodate markers
        fig.update_yaxes(range=[0, max_prob * 1.2], row=1, col=2)

# Attach callback to the scatter plot (trace 0)
fig.data[0].on_click(update_bar_chart)

fig

FigureWidget({
    'data': [{'customdata': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'KhAqICowKkAqUCpgKnAqgCqQKqAqsC'),
                             'dtype': 'i2'},
              'hovertemplate': 'Index %{customdata}<br>ΔEntropy=%{x:.3f}<br>Δp(Target)=%{y:.3f}<extra></extra>',
              'marker': {'color': 'steelblue', 'opacity': 0.7, 'size': 6},
              'mode': 'markers',
              'name': 'Data points',
              'type': 'scatter',
              'uid': 'b9d44676-d279-49a3-9e2b-ca4b0f42a5a4',
              'x': {'bdata': ('du5av6bhqL7mNZO+yA+xveCuyTxoKv' ... 'EAPngkyD18XTa+ePmIviCs2ryJutu+'),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('YpzFPhIiYzuE6Qw/1Y2OO8pSYT0xRj' ... 'enO+Wi1zoZoDw5u+vpNjnuqTzU/hk/'),
                    'dtype': 'f4'},
              'yaxis': 'y'},
             {'line': {'color': 'red'},
              'mode': 'lines',
              'name': 'Fitted line',
              'type': 'scatter',
   